- [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/c4dynamics/c4dynamics/blob/main/docs/source/programs/car_tracker_yolo11/car_tracker_yolo11.ipynb) ← Click to open in Google Colab
- To download this notebook, click the download icon in the toolbar above and select the .ipynb format.
- For any questions or comments, please open an issue on the [c4dynamics issues page](https://github.com/c4dynamics/c4dynamics/issues).


# Car Tracker – YOLO11 Detector and Kalman Filter

*A modern-detector refresh of the [Car Tracker – YOLOv3 example](https://c4dynamics.github.io/c4dynamics/programs/car_tracker/car_tracker.html).*

This notebook tracks a car through a video by pairing a **YOLO11** object detector with a
**Kalman filter**. It is a direct re-implementation of the original
[Car Tracker](https://c4dynamics.github.io/c4dynamics/programs/car_tracker/car_tracker.html)
use case: the estimation problem, the dynamic model, and the filtering strategy are
unchanged, only the detector is swapped from **YOLOv3** (served through OpenCV's DNN module
inside `c4dynamics`) to **YOLO11** (served through the `ultralytics` package).

A raw stream of detections is rarely good enough on its own: bounding boxes jitter from
frame to frame, and an object occasionally vanishes for a frame or two because of
occlusion or a missed detection. A Kalman filter fixes both problems by

- smoothing the bounding-box position and size, and
- predicting the object location on frames where the detector returns nothing.

As in the original example, two tracking modes are demonstrated:

- **Steady-state tracking** – a single precomputed Kalman gain is used for the whole run.
- **Adaptive-covariance tracking** – the measurement covariance is tightened around the
  moments where the car changes direction, so the filter trusts the detector more during
  manoeuvres.


## Architecture

The tracking pipeline is a single loop over the frames of a video. Every frame is pushed
through the detector; the detection that matches the class we care about (`car`) is handed
to the Kalman filter as a measurement; and the filter's smoothed state is what we draw on
screen. When the detector misses, the loop still runs — the filter's `predict` step alone
carries the estimate forward.

The cell below draws this flow as *Figure 1*.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

plt.style.use('dark_background')

fig, ax = plt.subplots(figsize=(11, 3.2), dpi=140)
ax.set_xlim(0, 22); ax.set_ylim(0, 6); ax.axis('off')

boxes = {
    'video' : (0.4,  'Video frames\n(1 / fps)'),
    'yolo'  : (4.2,  'YOLO11\ndetector'),
    'filt'  : (8.4,  "Class filter\n('car')"),
    'kf'    : (12.4, 'Kalman filter\npredict -> update'),
    'draw'  : (16.6, 'Draw bounding\nbox on frame'),
    'out'   : (20.2, 'Output\nvideo'),
}
w, h, y = 3.2, 1.8, 2.6
centers = {}
for key, (x, label) in boxes.items():
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.08',
                                linewidth=1.6, edgecolor='#4da6ff', facecolor='#12263a'))
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=9, color='white')
    centers[key] = (x + w/2, x, x + w)

order = ['video', 'yolo', 'filt', 'kf', 'draw', 'out']
for a, b in zip(order[:-1], order[1:]):
    ax.add_patch(FancyArrowPatch((centers[a][2], y + h/2), (centers[b][1], y + h/2),
                                 arrowstyle='-|>', mutation_scale=18, linewidth=1.4, color='#9ad'))

# feedback loop: next frame
ax.add_patch(FancyArrowPatch((centers['out'][0], y), (centers['video'][0], y),
                             connectionstyle='arc3,rad=0.28', arrowstyle='-|>',
                             mutation_scale=16, linewidth=1.2, color='#666'))
ax.text(11, 0.35, 'next frame', ha='center', va='center', fontsize=8, color='#999')

# predict-only branch note
ax.text(centers['kf'][0], y + h + 0.5, 'no detection  ->  predict only',
        ha='center', va='center', fontsize=8, color='#f0a')

plt.tight_layout()
plt.show()

<figcaption>Figure 1: Program flowchart. Each frame is read, run through the YOLO11
detector, filtered by class, and used to correct the Kalman estimate. On frames with no
valid <code>car</code> detection the filter runs its <code>predict</code> step only, and the
loop continues with the last estimate. The smoothed state is drawn back onto the frame and
written to the output video.</figcaption>

The role of `c4dynamics` here is the block on the right: the
[`kalman`](https://c4dynamics.github.io/c4dynamics/api/filters.kalman.html) class holds the
motion and measurement equations and runs `predict` / `update`, while the
[`pixelpoint`](https://c4dynamics.github.io/c4dynamics/api/states.lib.pixelpoint.html) state
class is the container that carries a detection (center pixel, bounding box, class label)
from the detector into the filter.

Let's build it step by step, following the standard structure: introduction, theoretical
background, dynamic model, sensors / inputs, filtering, the simulation loop, and results.

## Setup

The notebook is self-contained. On Google Colab the next cell installs the three
dependencies it needs:

- [`c4dynamics`](https://c4dynamics.github.io/c4dynamics/) – the Kalman filter and the
  `pixelpoint` state class,
- [`ultralytics`](https://docs.ultralytics.com/) – the YOLO11 model and weights,
- `opencv-python` – frame I/O and drawing.

Running locally, install the same packages once with
`pip install c4dynamics ultralytics opencv-python`.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q c4dynamics ultralytics opencv-python
    from google.colab.patches import cv2_imshow

In [ ]:
import cv2
import numpy as np
from IPython.display import Video
from matplotlib import pyplot as plt

from c4dynamics import pixelpoint, plotdefaults
from c4dynamics.filters import kalman
from c4dynamics import datasets

plt.style.use('dark_background')

### Video dataset

The clip is the same one used by the original example — a car drifting on a race track,
from a [free stock library](https://www.pexels.com/video/a-car-drifting-on-a-racing-track-4568686/).
The `c4dynamics` [datasets module](https://c4dynamics.github.io/c4dynamics/api/Datasets.html)
downloads it and returns a path to the cached file:

In [ ]:
video = datasets.video('drifting_car')

In [ ]:
Video(video, width = 640, height = 360, embed = True)

Open the stream with OpenCV and read its frame rate. The sampling interval between
frames, $dt = 1 / fps$, is the time step the Kalman filter propagates the state over:

In [ ]:
video_cap = cv2.VideoCapture(video)
fps = video_cap.get(cv2.CAP_PROP_FPS)
dt  = 1 / fps

width  = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f'frame size = {width} x {height} pixels,  fps = {fps:.1f},  dt = {dt:.4f} s')

A writer stores the annotated frames so the result can be played back at the end:

In [ ]:
vidout = cv2.VideoWriter('car_detected_yolo11.mp4',
                         int(video_cap.get(cv2.CAP_PROP_FOURCC)), fps, (width, height))

## Theoretical background

### YOLO11 object detection

**YOLO** (*You Only Look Once*) is a family of single-stage detectors: a single forward
pass of a convolutional network turns an image into a set of bounding boxes, each with a
class label and a confidence score. There is no separate region-proposal stage, which is
what makes the family fast enough for video.

**YOLO11** (2024) is the current generation from Ultralytics. Compared with **YOLOv3**
(2018), which the original notebook used, it keeps the same input/output contract — image
in, boxes out — but differs in ways that matter for tracking:

| | YOLOv3 | YOLO11 |
|---|---|---|
| detection head | anchor-based | anchor-free |
| backbone / neck | Darknet-$53$ | improved $CSP$ with $C3k2$ / $C2PSA$ blocks |
| smallest model | $\approx 62$ M parameters | $\approx 2.6$ M parameters (`yolo11n`) |
| served in this notebook via | OpenCV $DNN$ (`c4dynamics.detectors.yolov3`) | `ultralytics` package |

For our purpose the important consequence is that YOLO11-nano is both smaller and more
accurate than YOLOv3, so the raw detections fed to the filter are already tighter and
jitter less. The tracking machinery around it is identical.

Both models are trained on the **COCO** dataset and predict the same $80$ classes
(`person`, `bicycle`, `car`, `bus`, `truck`, …). We only use one of them, `car`.

<div style='text-align: center;'>
  <img src='../../_architecture/yolo-object-detection.jpg' alt='Object detection with YOLO: three bounding boxes labelled dog, bicycle, and truck are drawn over a single photograph, each produced in one forward pass of the network.'>
  <figcaption>Figure 2: One-stage detection. A single pass of the network localizes and
  classifies every object in the frame at once — here the COCO classes
  <code>dog</code>, <code>bicycle</code>, and <code>truck</code>. In this notebook we run
  the same step with YOLO11 and keep only the <code>car</code> box.</figcaption>
</div>

### Kalman filtering

A Kalman filter is the optimal linear estimator for a system driven by Gaussian noise.
It maintains a state estimate and its covariance, and updates them recursively in two
steps per frame:

- **Predict** – propagate the state with the equations of motion, $x_{k+1} = F \cdot x_k$,
  and grow the covariance by the process noise $Q$. This step runs on *every* frame.
- **Update** – when a detection $z_k$ is available, blend it with the prediction using the
  Kalman gain $K$, which weighs the process noise $Q$ against the measurement noise $R$:
  $$ x_k \leftarrow x_k + K \cdot (z_k - H \cdot x_k) $$

The gain $K$ is what makes the filter adaptive: when $R$ is small the estimate follows the
detector; when $R$ is large it follows the motion model. We exploit exactly this in the
second tracking mode.

To set the filter up we need three ingredients: a **dynamic model** ($F$, $Q$), a
**measurement model** ($H$, $R$), and initial conditions.

## Dynamic model

Assume the car moves at approximately **constant velocity** between frames and that its
bounding box keeps a roughly **fixed size**. In discrete time the motion is a set of
first-order difference equations:

$$
  {x_c}_{k+1} = {x_c}_k + {v_x}_k \cdot dt \\
  {y_c}_{k+1} = {y_c}_k + {v_y}_k \cdot dt \\
  w_{k+1} = w_k    \\
  h_{k+1} = h_k    \\
  {v_x}_{k+1} = {v_x}_k  \\
  {v_y}_{k+1} = {v_y}_k
$$

Where:

- $x_c, y_c$ are the coordinates of the bounding-box center pixel,
- $w, h$ are the bounding-box width and height in pixels,
- $v_x, v_y$ are the object velocities in $pixel / second$,
- $dt = 1 / fps$ is the time between frames,
- $k$ is the discrete frame index.

In state-space form:

$$ x_{k+1} = F \cdot x_k + \eta_k $$

with the state vector $x = [x_c, y_c, w, h, v_x, v_y]^T$ (order $n = 6$) and the state
transition matrix

$$
F = \begin{bmatrix}
        1 & 0 & 0 & 0 & dt & 0 \\
        0 & 1 & 0 & 0 & 0 & dt \\
        0 & 0 & 1 & 0 & 0 & 0 \\
        0 & 0 & 0 & 1 & 0 & 0 \\
        0 & 0 & 0 & 0 & 1 & 0 \\
        0 & 0 & 0 & 0 & 0 & 1
      \end{bmatrix}
$$

$\eta_k$ is zero-mean white process noise with covariance $Q$; it absorbs everything the
constant-velocity assumption leaves out (mainly the accelerations during the drift).

In [ ]:
# process dynamics: constant velocity, constant box size
F = np.eye(6)
F[0, 4] = F[1, 5] = dt
print(F)

## Sensors / inputs

The only sensor is the **YOLO11 detector**. Each call returns, for every detected
object, a bounding box $[x_c, y_c, w, h]$ and a class label. We wrap each detection in a
`pixelpoint` — the `c4dynamics` state class for "a point in an image with a bounding box" —
so it plugs straight into the filter's `update`.

In [ ]:
from c4dynamics import pixelpoint
print(pixelpoint())          # state layout: [ x  y  w  h ]

Load YOLO11-nano. The weights (`yolo11n.pt`, a few $MB$) download automatically on first
use and are then cached by `ultralytics`:

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')

A thin adapter turns one frame into a list of `pixelpoint` detections. It mirrors the
`detect` method of `c4dynamics.detectors.yolov3` used in the original notebook, so the main
loop below is identical to that example:

In [ ]:
def detect(frame):
    """Run YOLO11 on a BGR frame, return a list of pixelpoint detections."""
    result = model(frame, verbose = False)[0]
    fsize  = (frame.shape[1], frame.shape[0])

    points = []
    for box in result.boxes:
        xc, yc, w, h = box.xywh[0].tolist()
        pp = pixelpoint(x = xc, y = yc, w = w, h = h)
        pp.fsize    = fsize
        pp.class_id = model.names[int(box.cls)]
        points.append(pp)
    return points

`pixelpoint.X` is the vector $[x_c, y_c, w, h]^T$ — exactly the measurement the filter
expects. The `class_id` attribute is used to reject everything that is not a `car`.

### Measurement model

The detector observes the first four state variables directly and says nothing about
velocity, so the measurement equation is

$$ z_k = H \cdot x_k + \nu_k $$

with

$$
  H = \begin{bmatrix}
        1 & 0 & 0 & 0 & 0 & 0 \\
        0 & 1 & 0 & 0 & 0 & 0 \\
        0 & 0 & 1 & 0 & 0 & 0 \\
        0 & 0 & 0 & 1 & 0 & 0
      \end{bmatrix}
$$

$\nu_k$ is zero-mean measurement noise with covariance $R$.

In [ ]:
H = np.zeros((4, 6))
H[range(4), range(4)] = 1
print(H)

The detector's true error is hard to pin down exactly. Following the original example
we set the one-sigma measurement noise to a small fraction of the image width, and start
with a process noise $Q$ of the same magnitude so that neither the model nor the detector
is favoured a priori:

$$ std_{measure} = \frac{im_{width}}{10000} $$

In [ ]:
measure_std = width / 10000
R = np.eye(4) * measure_std**2    # measurement covariance
Q = np.eye(6) * measure_std**2    # process covariance

## Filtering

### Observability

Before trusting the filter, check that the measurements actually constrain the whole
state. The system is observable if the observability matrix

$$
  \mathcal{O} = \begin{bmatrix}
        H \\ H \cdot F \\ H \cdot F^2 \\ \vdots \\ H \cdot F^{\,n-1}
      \end{bmatrix}
$$

has rank equal to the system order $n = 6$.

In [ ]:
O = H
n = len(F)
for i in range(1, n):
    O = np.vstack((O, H @ np.linalg.matrix_power(F, i)))
rank = np.linalg.matrix_rank(O)
print(f'rank(O) = {rank},  n = {n}  ->  '
      + ('observable' if rank == n else 'NOT observable'))

The velocities are observable even though they are never measured directly: two
successive position measurements pin them down through $F$.

### Steady-state Kalman gain

$F$, $H$, $Q$, $R$ are all constant, so the system is **linear time-invariant**. The
covariance $P$ and the gain $K$ then converge to fixed values and can be computed once, up
front, instead of every frame. The `kalman` class does this when `steadystate = True`:

In [ ]:
kf = kalman({'x': 0, 'y': 0, 'w': 0, 'h': 0, 'vx': 0, 'vy': 0},
            F = F, H = H, Q = Q, R = R, steadystate = True)
print(kf._Kinf)

Each measured variable is corrected by roughly $0.6$ of the innovation — the filter
splits the difference between prediction and detection, because we made $Q$ and $R$ equal.

Two helpers convert the state vector into the pixel corners OpenCV needs to draw a
rectangle:

In [ ]:
# top-left and bottom-right corners of the bounding box
def tl(X): return int(X[0] - X[2] / 2), int(X[1] - X[3] / 2)
def br(X): return int(X[0] + X[2] / 2), int(X[1] + X[3] / 2)

## Simulation / main loop

### Mode 1 – steady state

The loop is the pipeline from *Figure 1*. `predict` runs on every frame; `update` runs
only when a `car` is detected. `kf.store(t)` records the state history for the plots, and
`kf.storeparams('detect', t)` keeps the raw detection alongside it for comparison.

In [ ]:
t = 0
video_cap = cv2.VideoCapture(video)
snapshot = None

while video_cap.isOpened():
    kf.store(t)
    kf.predict()

    ret, frame = video_cap.read()
    if not ret:
        break

    # take only the first 'car' detection, if any
    d = next(iter([di for di in detect(frame) if di.class_id == 'car']), None)
    if d:
        kf.update(d.X)
        kf.detect = d
        kf.storeparams('detect', t)

    cv2.rectangle(frame, tl(kf.X), br(kf.X), [0, 255, 0], 2)

    if abs(t - 2.0) < dt:            # keep one frame for Figure 3
        snapshot = frame.copy()

    if IN_COLAB:
        cv2_imshow(frame)
    else:
        cv2.imshow('', frame)
        cv2.waitKey(10)

    vidout.write(frame)
    t += dt

video_cap.release()
vidout.release()
cv2.destroyAllWindows()

In [ ]:
Video('car_detected_yolo11.mp4', width = 640, height = 360, embed = True)

The green box is the **Kalman estimate**, not the raw detection. *Figure 3* shows one
frame on its own:

In [ ]:
if snapshot is not None:
    plt.figure(figsize = (6, 3.4), dpi = 140)
    plt.imshow(cv2.cvtColor(snapshot, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('YOLO11 + Kalman, steady-state mode  (t = 2 s)', fontsize = 9)
    plt.show()

<figcaption>Figure 3: A single annotated frame at $t = 2\,s$. The rectangle is the
filtered state $[x_c, y_c, w, h]$ drawn back onto the image. Because the estimate is
smoothed, the box stays steady even on frames where the detector's raw output would have
jumped.</figcaption>

### Results – steady state

`plot_track` draws the estimated trajectory on the image plane, optionally with the raw
detections and with timestamps:

- `title` (`str`) – title text,
- `detections` (`bool`) – overlay the raw YOLO11 detections,
- `printtime` (`bool`) – annotate key $x$ positions with their timestamp,
- `axislim` (`list`) – axis window $[x_{min}, x_{max}, y_{min}, y_{max}]$.

In [ ]:
def plot_track(kf, title, detections = False, printtime = False, axislim = None):

    _, ax = plt.subplots(1, 1, dpi = 200, figsize = (4, 2.25),
                         gridspec_kw = {'left': 0.15, 'right': .9, 'top': .9, 'bottom': .2})
    ax.plot(kf.data('x')[1], kf.data('y')[1], 'om', markersize = 1, label = 'estimation')

    if detections:
        dxdy = np.vectorize(lambda d: (d.x, d.y) if isinstance(d, pixelpoint) else np.nan)(kf.data('detect')[1])
        ax.plot(dxdy[0], dxdy[1], 'co', markersize = .5, label = 'detection')

    plotdefaults(ax, title, 'X', 'Y', 8)
    ax.legend(fontsize = 6, facecolor = None)
    ax.invert_yaxis()

    if printtime:
        time   = kf.data('t')
        xgrid  = np.arange(0, width, 100)
        xdata  = kf.data('x')[1]
        txgrid = [time[np.argmin(np.abs(xdata - v))] for v in xgrid]
        yxgrid = [kf.timestate(txi)[1] for txi in txgrid]
        for pt in zip(txgrid, xgrid, yxgrid):
            ax.text(pt[1], pt[2], f'{pt[0]:.2f}', fontsize = 5, color = "white")

    if axislim:
        ax.axis(axislim)

In [ ]:
plot_track(kf, title = 'Steady-State Mode  (YOLO11)')

The path is clearly **not** a straight line, but it breaks into roughly three nearly
linear segments — which is why a constant-velocity model works as well as it does here.

In [ ]:
plot_track(kf, title = 'Steady-State Mode  (YOLO11)', detections = True)

Overlaying the raw detections (cyan) on the estimate (magenta): the two agree along the
straight segments. Zooming into the first corner, near $x \approx 500$, $t \approx 4\,s$:

In [ ]:
plot_track(kf, title = 'Steady-State Mode  (YOLO11)', detections = True,
           axislim = [400, 600, 250, 400])
pt = [490, 315]
circle = plt.Circle(pt, 15, color = 'blue', fill = False, linewidth = 1)
plt.gca().add_patch(circle)
plt.gca().invert_yaxis()

Inside the circle the magenta estimate lags the cyan detections. With $Q$ and $R$
equal, the filter trusts its constant-velocity prediction as much as the measurement and
averages the two — so during a turn, where the constant-velocity model is wrong, the
estimate is pulled off the true path. In steady-state mode there is no way to fix this
locally.

### Mode 2 – adaptive covariance

A time-invariant filter cannot use a precomputed gain if $R$ changes with time. Drop
`steadystate` and let the filter carry its covariance $P$ frame to frame, initialised at
$P_0 = Q$:

In [ ]:
kf = kalman({'x': 0, 'y': 0, 'w': 0, 'h': 0, 'vx': 0, 'vy': 0},
            P0 = Q, F = F, H = H, Q = Q, R = R)

From the previous plot the direction changes happen around $t \approx 4\,s$ and
$t \approx 7.5\,s$. Near those instants we shrink the measurement standard deviation by a
factor of $10$, so the filter leans on the detector rather than the (locally wrong) motion
model:

$$
  std_{measure} =
  \begin{cases}
      im_{width} / 100000  & t \approx 4\,s,\ 7.5\,s \\
      im_{width} / 10000   & \text{otherwise}
  \end{cases}
$$

In [ ]:
t_transitions = [4, 7.5]

The loop is the steady-state loop with one addition — recompute $R$ per frame and pass
it to `update`:

In [ ]:
t = 0
video_cap = cv2.VideoCapture(video)

while video_cap.isOpened():
    kf.store(t)
    kf.predict()

    ret, frame = video_cap.read()
    if not ret:
        break

    d = next(iter([di for di in detect(frame) if di.class_id == 'car']), None)
    if d:
        if np.isclose(t, t_transitions, atol = 0.2).any():
            measure_std = width / 100000
        else:
            measure_std = width / 10000
        R = np.eye(4) * measure_std**2

        kf.update(d.X, R = R)
        kf.detect = d
        kf.storeparams('detect', t)

    cv2.rectangle(frame, tl(kf.X), br(kf.X), [0, 255, 0], 2)

    if IN_COLAB:
        cv2_imshow(frame)
    else:
        cv2.imshow('', frame)
        cv2.waitKey(10)

    t += dt

video_cap.release()
cv2.destroyAllWindows()

### Results – adaptive covariance

Same corner as before, near $t \approx 4\,s$:

In [ ]:
plot_track(kf, title = 'Adaptive-Covariance Mode  (YOLO11)', detections = True,
           axislim = [400, 600, 250, 400])
circle = plt.Circle(pt, 15, color = 'blue', fill = False, linewidth = 1)
plt.gca().add_patch(circle)
plt.gca().invert_yaxis()

The estimate now hugs the detections through the turn: with $R$ small, the filter
follows the measurement where the motion model cannot be trusted.

In [ ]:
plot_track(kf, title = 'Adaptive-Covariance Mode  (YOLO11)', detections = True)

And over the full run the trajectory tracks the car through all three segments and both
corners.

## Summary

This notebook re-implements the [Car Tracker](https://c4dynamics.github.io/c4dynamics/programs/car_tracker/car_tracker.html)
use case with a **YOLO11** detector in place of **YOLOv3**.

- The detector is loaded from `ultralytics` (`yolo11n.pt`) and wrapped by a small `detect`
  adapter that returns `c4dynamics` `pixelpoint` objects — center pixel, bounding box, and
  class label — so the rest of the pipeline is byte-for-byte the original.
- The tracked state is $x = [x_c, y_c, w, h, v_x, v_y]^T$, propagated with a
  constant-velocity, constant-box-size model ($F$, $Q$) and corrected by the detector
  ($H$, $R$).
- The system is $LTI$ and observable (rank $\mathcal{O} = 6$), so it runs in
  **steady-state** mode with a single precomputed gain $K$.
- Steady state averages prediction and measurement, which lags the estimate during turns.
  **Adaptive-covariance** mode fixes this by shrinking $R$ by $10\times$ near
  $t \approx 4\,s$ and $t \approx 7.5\,s$, letting the detector lead through the corners.

Swapping in a newer detector changed one function; the estimation and filtering logic — the
`c4dynamics` part — did not move. That separation is the point.